In [14]:
from datasets import load_dataset

# 加载 Hugging Face 上的指定数据集的 train 划分
dataset = load_dataset("/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA-Corpus", split="train")
# 随机打乱（可选），然后选取前 1000 条
subset = dataset.shuffle(seed=42).select(range(5000))

# 保存子集为 Parquet，路径可以是文件或目录
subset.to_parquet("OpenDocVQA-Corpus/subset5000.parquet")


Creating parquet from Arrow format: 100%|██████████| 50/50 [00:05<00:00,  9.89ba/s]


2146640336

In [15]:
from datasets import load_dataset

# 从本地 Parquet 文件加载为 DatasetDict 或 Dataset
reloaded = load_dataset("parquet", data_files="OpenDocVQA-Corpus/subset5000.parquet")


Generating train split: 5000 examples [00:06, 750.69 examples/s]


In [ ]:

from PIL import Image
import time

# reloaded['train'][0]['image'].show()
def add_label(example,idx):
    print(idx)

    example["my_new_label"] = 1
    # time.sleep(0.5)
    return example
subset = subset.map(add_label,with_indices=True,) 
print(subset[0])

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
QA_data = load_dataset("/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA", split="train")
curpus_data = load_dataset("parquet", data_files="/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA-Corpus_5000/subset5000.parquet")['train']
doc_ids=set()
for i in tqdm(range(len(curpus_data))):
    doc_ids.add(curpus_data[i]['doc_id'])
print(len(doc_ids))
# print(QA_data[0])

{'query_id': 'visualmrc-train_0', 'query': "Instruct: I'm looking for a screenshot image that answers the question.\nQuery: What is the most commonly spoken language in Connecticut", 'answers': ['AS with the rest of the United States the most commonly spoken language in Connecticut is English'], 'relevant_doc_ids': ['visualmrc/wikitravel.org/en__Connecticut02.png'], 'dataset_names': ['visualmrc']}


In [21]:
import copy
QA_data_copy=copy.deepcopy(QA_data)
def filter_by_doc_id(example,doc_ids):

    return example['relevant_doc_ids'][0] in doc_ids
print('before filter',len(QA_data))
QA_data_copy=QA_data_copy.filter(filter_by_doc_id,fn_kwargs={'doc_ids':doc_ids})
print('after filter',len(QA_data_copy))
print(QA_data_copy[0])
num_dict={doc_id:0 for doc_id in doc_ids}
for i in range(len(QA_data_copy)):
    doc_id= QA_data_copy[i]['relevant_doc_ids'][0]
    if doc_id in doc_ids:
        num_dict[doc_id]+=1
print(max(num_dict.values()))
print(min(num_dict.values()))
count=0
for k,v in num_dict.items():
    if v==0:
        count+=1
print(count)

before filter 41020
after filter 16073
{'query_id': 'infovqa-train_0', 'query': 'Instruct: Given a question, retrieve an infographic to answer the question.\nQuery: Which is the largest mammal in North America', 'answers': ['American Bison'], 'relevant_doc_ids': ['infovqa/42467.jpeg'], 'dataset_names': ['infovqa']}
211
0
12


In [22]:
QA_data_copy.to_parquet("/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA_5000")

Creating parquet from Arrow format: 100%|██████████| 17/17 [00:00<00:00, 20.49ba/s]


4589883

In [ ]:
from datasets import load_dataset
train_data = load_dataset(
    "parquet",
    None,
    data_files="/home/zhuyinglian/fdu02_dir/zyl/downloads/OpenDocVQA_5000/OpenDocVQA_5000.parquet",
    split='train',
    cache_dir=None,
)
corpus  = load_dataset(
                'parquet',
                None,
                data_files='/home/zhuyinglian/fdu02_dir/zyl/downloads/OpenDocVQA-Corpus_5000/subset5000.parquet',
                split='train',
                cache_dir=None,
)
len(train_data)
